# Deep Learning with PyTorch · Convolutional neural networks (CNNs)

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Convolution arithmetic

In [ ]:
from torch import nn
x = torch.randn(1, 1, 28, 28)
for k, s, p in [(3, 1, 1), (3, 2, 1), (5, 1, 0), (7, 2, 3)]:
    out = nn.Conv2d(1, 4, k, stride=s, padding=p)(x)
    formul = (28 + 2 * p - (k - 1) - 1) // s + 1
    print(f'k={k} s={s} p={p}: {tuple(out.shape)} formula={formul}')            # same as the formula

## 2. Model and MNIST

In [ ]:
from torch import nn
class KucukCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 7 * 7, 10)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))                     # (N, 8, 14, 14)
        x = self.pool(torch.relu(self.conv2(x)))                     # (N, 16, 7, 7)
        return self.fc(x.flatten(1))                                 # (N, 10)
print(sum(p.numel() for p in KucukCNN().parameters()))

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
tr = datasets.MNIST('veri', train=True, download=True, transform=tf)
te = datasets.MNIST('veri', train=False, download=True, transform=tf)
tr_dl = DataLoader(tr, batch_size=64, shuffle=True); te_dl = DataLoader(te, batch_size=1000)
print(len(tr), len(te))

In [ ]:
def evaluate(model):
    model.eval(); ok = 0
    with torch.no_grad():
        for xb, yb in te_dl:
            ok += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return ok / len(te)

model = KucukCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(4):
    model.train()
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); nn.functional.cross_entropy(model(xb), yb).backward(); opt.step()
    print(f'epoch {epoch + 1}: test acc {evaluate(model):.4f}')

## 3. Feature maps

In [ ]:
import matplotlib.pyplot as plt
maps = {}
model.conv1.register_forward_hook(lambda m, i, o: maps.__setitem__('c1', o.detach().cpu()))
img, label = te[0]
model(img.unsqueeze(0).to(device))
fig, ax = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    ax[0, i].imshow(model.conv1.weight[i, 0].detach().cpu(), cmap='bwr'); ax[0, i].axis('off')
    ax[1, i].imshow(maps['c1'][0, i], cmap='magma'); ax[1, i].axis('off')
plt.suptitle(f'kernels and maps, label: {label}'); plt.show()

## Exercises
1. Double the channel counts; how did parameters and accuracy change?
2. Add `nn.BatchNorm2d`.
3. Show 10 misclassified images.